# Metacatalog query browser

Browse Parquet catalogs under `CATALOG_DIR` (per-hour sources, LST-merged bands,
and the global metacatalog), pick one table, inspect it in a sortable/filterable
grid, and find the nearest source to a sky coordinate.

Launch with `jupyter lab` (or your usual kernel that has `lwa-catalog[viz]`).
**Run cells in order.**


In [6]:
from __future__ import annotations

import re
from pathlib import Path

import astropy.units as u
import numpy as np
import pandas as pd
import panel as pn
import param
from astropy.coordinates import SkyCoord

from lwa_catalog.io import read_table

# --- user configuration ---------------------------------------------------
CATALOG_DIR = Path("/fast/claw/metacatalog")

TABLE_HEIGHT = 420
TABLE_PAGE_SIZE = 50
DEFAULT_N_NEAREST = 5
MAX_TABLE_COLUMNS = 40  # keep Tabulator responsive for wide metacatalog rows

# Preferred column order when present (others follow alphabetically)
PREFERRED_COLUMNS = (
    "meta_id",
    "Source_id",
    "RA",
    "DEC",
    "Peak_flux",
    "Total_flux",
    "band",
    "lst_hour",
    "bands_present",
    "origin_band",
    "n_lst_contributions",
    "source_file",
)

pn.extension("tabulator", throttled=True)


## Available catalog files

Inventory of every `*.parquet` under `CATALOG_DIR`, classified by artifact kind.


In [7]:
_SOURCES_RE = re.compile(r"^sources_(?P<lst>\d+h)_(?P<band>Full|Blue|Green|Red)\.parquet$")
_LST_RE = re.compile(r"^metacatalog_lst_(?P<band>Full|Blue|Green|Red)\.parquet$")


def classify_catalog(path: Path) -> dict:
    """Return kind / LST / band metadata for a catalog Parquet path."""
    name = path.name
    if name == "metacatalog.parquet":
        return {"kind": "metacatalog", "lst_hour": None, "band": None}
    m = _LST_RE.match(name)
    if m:
        return {"kind": "lst_merged", "lst_hour": None, "band": m.group("band")}
    m = _SOURCES_RE.match(name)
    if m:
        return {"kind": "sources", "lst_hour": m.group("lst"), "band": m.group("band")}
    return {"kind": "other", "lst_hour": None, "band": None}


def inventory_catalogs(catalog_dir: Path) -> pd.DataFrame:
    """List Parquet catalogs with size and row counts (metadata only)."""
    rows: list[dict] = []
    for path in sorted(catalog_dir.glob("*.parquet")):
        meta = classify_catalog(path)
        try:
            import pyarrow.parquet as pq

            n_rows = pq.ParquetFile(path).metadata.num_rows
        except Exception:
            n_rows = None
        rows.append(
            {
                "file": path.name,
                "kind": meta["kind"],
                "lst_hour": meta["lst_hour"],
                "band": meta["band"],
                "n_rows": n_rows,
                "size_mb": round(path.stat().st_size / 1e6, 2),
                "path": str(path),
            }
        )
    return pd.DataFrame(rows)


catalog_index = inventory_catalogs(CATALOG_DIR)
print(f"{len(catalog_index)} Parquet files in {CATALOG_DIR}")
print(catalog_index["kind"].value_counts().to_string())

inventory_table = pn.widgets.Tabulator(
    catalog_index.drop(columns=["path"]),
    pagination="local",
    page_size=25,
    height=320,
    sizing_mode="stretch_width",
    layout="fit_data_table",
    header_filters=True,
    show_index=False,
    disabled=True,
    sortable=True,
    selectable=1,
)
inventory_table


85 Parquet files in /fast/claw/metacatalog
kind
sources        80
lst_merged      4
metacatalog     1


Tabulator(disabled=True, header_filters=True, height=320, page_size=25, pagination='local', show_index=False, sizing_mode='stretch_width', value=              ...)

## Select a table, browse rows, find nearest source

1. Click a row in the inventory table above, **or** use the Kind / LST / Band / file selectors.
2. The main table is sortable (click headers) and filterable (header search boxes).
3. Enter a coordinate as decimal degrees (`123.4 -12.3`) or sexagesimal
   (`08h25m36s +20d45m12s` / `12:30:00 +45:00:00`) and click **Find nearest**.


In [8]:
def order_columns(df: pd.DataFrame) -> list[str]:
    preferred = [c for c in PREFERRED_COLUMNS if c in df.columns]
    rest = sorted(c for c in df.columns if c not in preferred)
    cols = preferred + rest
    if len(cols) > MAX_TABLE_COLUMNS:
        cols = cols[:MAX_TABLE_COLUMNS]
    return cols


def parse_coordinate(text: str) -> SkyCoord:
    """Parse a single sky position from free-form text."""
    text = text.strip()
    if not text:
        raise ValueError("Coordinate string is empty")

    # Two decimal numbers → degrees
    parts = text.replace(",", " ").split()
    if len(parts) == 2:
        try:
            ra = float(parts[0])
            dec = float(parts[1])
            return SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs")
        except ValueError:
            pass

    # Astropy flexible parsers (sexagesimal, unit strings, …)
    try:
        return SkyCoord(text, unit=(u.hourangle, u.deg), frame="icrs")
    except Exception:
        return SkyCoord(text, frame="icrs")


def nearest_sources(
    df: pd.DataFrame,
    coord: SkyCoord,
    *,
    n: int = 5,
    ra_col: str = "RA",
    dec_col: str = "DEC",
) -> pd.DataFrame:
    """Return the ``n`` catalog rows closest to ``coord`` (great-circle)."""
    if ra_col not in df.columns or dec_col not in df.columns:
        raise KeyError(f"Need {ra_col!r} and {dec_col!r} columns for sky matching")
    if df.empty:
        raise ValueError("Catalog is empty")

    catalog = SkyCoord(
        ra=np.asarray(df[ra_col], dtype=float) * u.deg,
        dec=np.asarray(df[dec_col], dtype=float) * u.deg,
        frame="icrs",
    )
    seps = coord.separation(catalog)
    order = np.argsort(seps.deg)
    n = max(1, min(int(n), len(df)))
    out = df.iloc[order[:n]].copy()
    out.insert(0, "sep_arcmin", np.round(seps.deg[order[:n]] * 60.0, 4))
    out.insert(1, "sep_deg", np.round(seps.deg[order[:n]], 6))
    return out.reset_index(drop=True)


class CatalogBrowser(pn.viewable.Viewer):
    """Interactive browser for Parquet catalogs under ``CATALOG_DIR``."""

    kind = param.Selector(default="all", objects=["all"], doc="Artifact kind filter")
    lst_hour = param.Selector(default="all", objects=["all"], doc="LST hour filter")
    band = param.Selector(default="all", objects=["all"], doc="Band filter")
    catalog_file = param.Selector(default="", objects=[""], doc="Selected Parquet file")
    coordinate = param.String(
        default="83.633 -5.391",
        doc="Query coordinate (decimal deg or sexagesimal)",
    )
    n_nearest = param.Integer(default=DEFAULT_N_NEAREST, bounds=(1, 50), doc="Neighbors to return")

    def __init__(self, index: pd.DataFrame, **params):
        # Validate Selector values only after objects lists are populated.
        params = dict(params)
        params.pop("kind", None)
        params.pop("lst_hour", None)
        params.pop("band", None)
        params.pop("catalog_file", None)
        super().__init__(**params)

        self._ready = False
        self._index = index.copy()
        self._df: pd.DataFrame | None = None
        self._status = pn.pane.Markdown("", sizing_mode="stretch_width")
        self._nearest_status = pn.pane.Markdown("", sizing_mode="stretch_width")

        kinds = ["all", *sorted(str(x) for x in index["kind"].dropna().unique())]
        self.param.kind.objects = kinds

        with pn.config.set(sizing_mode="stretch_width"):
            self._kind_w = pn.widgets.Select.from_param(self.param.kind, name="Kind")
            self._lst_w = pn.widgets.Select.from_param(self.param.lst_hour, name="LST hour")
            self._band_w = pn.widgets.Select.from_param(self.param.band, name="Band")
            self._file_w = pn.widgets.Select.from_param(self.param.catalog_file, name="Catalog file")
            self._coord_w = pn.widgets.TextInput.from_param(
                self.param.coordinate, name="Coordinate", placeholder="RA Dec"
            )
            self._n_w = pn.widgets.IntInput.from_param(self.param.n_nearest, name="N nearest")
            self._find_btn = pn.widgets.Button(name="Find nearest", button_type="primary")
            self._find_btn.on_click(self._on_find_nearest)

            self._table = pn.widgets.Tabulator(
                pd.DataFrame(),
                pagination="local",
                page_size=TABLE_PAGE_SIZE,
                height=TABLE_HEIGHT,
                sizing_mode="stretch_width",
                layout="fit_data_table",
                header_filters=True,
                show_index=False,
                disabled=True,
                sortable=True,
            )
            self._nearest_table = pn.widgets.Tabulator(
                pd.DataFrame(),
                pagination=None,
                height=220,
                sizing_mode="stretch_width",
                layout="fit_data_table",
                show_index=False,
                disabled=True,
                sortable=True,
            )

            self._panel = pn.Column(
                pn.Row(self._kind_w, self._lst_w, self._band_w),
                self._file_w,
                self._status,
                self._table,
                pn.pane.Markdown("### Nearest-source query", disable_anchors=True),
                pn.Row(self._coord_w, self._n_w, self._find_btn),
                self._nearest_status,
                self._nearest_table,
            )

        self._ready = True
        self.kind = "metacatalog" if "metacatalog" in kinds else "all"
        self._sync_filters()
        self._load_selected()

    def __panel__(self):
        return self._panel

    def select_file(self, filename: str) -> None:
        """Select a catalog by filename (updates kind / LST / band filters)."""
        rows = self._index.loc[self._index["file"] == filename]
        if rows.empty:
            raise KeyError(f"Unknown catalog file: {filename}")
        row = rows.iloc[0]
        kind = str(row["kind"])
        if kind not in self.param.kind.objects:
            self.param.kind.objects = ["all", *sorted(set(self.param.kind.objects) | {kind})]
        self.kind = kind
        self._sync_filters()
        if filename not in self.param.catalog_file.objects:
            self.param.catalog_file.objects = [*self.param.catalog_file.objects, filename]
        self.catalog_file = filename
        self._load_selected()

    def _filtered_index(self) -> pd.DataFrame:
        df = self._index
        if self.kind != "all":
            df = df[df["kind"] == self.kind]
        if self.lst_hour != "all":
            df = df[df["lst_hour"] == self.lst_hour]
        if self.band != "all":
            df = df[df["band"] == self.band]
        return df

    @param.depends("kind", watch=True, on_init=False)
    def _on_kind_change(self) -> None:
        if not getattr(self, "_ready", False):
            return
        self._sync_filters()

    @param.depends("lst_hour", "band", watch=True, on_init=False)
    def _on_subfilter_change(self) -> None:
        if not getattr(self, "_ready", False):
            return
        self._sync_file_options(reset_if_missing=True)

    @param.depends("catalog_file", watch=True, on_init=False)
    def _on_file_change(self) -> None:
        if not getattr(self, "_ready", False):
            return
        self._load_selected()

    def _sync_filters(self) -> None:
        base = self._index if self.kind == "all" else self._index[self._index["kind"] == self.kind]
        lst_opts = ["all", *sorted(str(x) for x in base["lst_hour"].dropna().unique())]
        band_opts = ["all", *sorted(str(x) for x in base["band"].dropna().unique())]
        self.param.lst_hour.objects = lst_opts
        self.param.band.objects = band_opts
        if self.lst_hour not in lst_opts:
            self.lst_hour = "all"
        if self.band not in band_opts:
            self.band = "all"
        self._sync_file_options(reset_if_missing=True)

    def _sync_file_options(self, *, reset_if_missing: bool) -> None:
        files = self._filtered_index()["file"].tolist() or [""]
        self.param.catalog_file.objects = files
        if files == [""]:
            self.catalog_file = ""
            return
        if reset_if_missing and self.catalog_file not in files:
            preferred = ["metacatalog.parquet"]
            preferred += [f for f in files if f.endswith("_Full.parquet")]
            self.catalog_file = next((f for f in preferred if f in files), files[0])

    def _load_selected(self) -> None:
        if not self.catalog_file:
            self._df = None
            self._table.value = pd.DataFrame()
            self._status.object = "_No catalog matches the current filters._"
            return

        path = CATALOG_DIR / self.catalog_file
        df = read_table(path, as_pandas=True)
        self._df = df
        cols = order_columns(df)
        self._table.value = df[cols]
        skipped = len(df.columns) - len(cols)
        extra = f" (showing {len(cols)}/{len(df.columns)} columns)" if skipped else ""
        self._status.object = (
            f"**{self.catalog_file}** — {len(df):,} rows × {len(df.columns)} columns{extra}. "
            "Click headers to sort; use header filters to narrow rows."
        )
        self._nearest_table.value = pd.DataFrame()
        self._nearest_status.object = ""

    def _on_find_nearest(self, _event=None) -> None:
        if self._df is None or self._df.empty:
            self._nearest_status.object = "**Load a catalog first.**"
            return
        try:
            coord = parse_coordinate(self.coordinate)
            hits = nearest_sources(self._df, coord, n=self.n_nearest)
        except Exception as exc:
            self._nearest_status.object = f"**Query failed:** `{exc}`"
            self._nearest_table.value = pd.DataFrame()
            return

        cols = ["sep_arcmin", "sep_deg", *order_columns(hits.drop(columns=["sep_arcmin", "sep_deg"]))]
        self._nearest_table.value = hits[[c for c in cols if c in hits.columns]]
        nearest = hits.iloc[0]
        self._nearest_status.object = (
            f"Query `{coord.to_string('hmsdms')}` → nearest at "
            f"**{nearest['sep_arcmin']:.3f} arcmin** "
            f"(RA={nearest['RA']:.6f}, Dec={nearest['DEC']:.6f}). "
            f"Showing top {len(hits)}."
        )


browser = CatalogBrowser(catalog_index)


def _on_inventory_select(event) -> None:
    if not inventory_table.selection:
        return
    row = inventory_table.value.iloc[int(inventory_table.selection[0])]
    browser.select_file(str(row["file"]))


inventory_table.param.watch(_on_inventory_select, "selection")
browser


CatalogBrowser(band='all', catalog_file='metacatalog.parquet', coordinate='83.633 -5.391', kind='metacatalog', lst_hour='all', n_nearest=5, name='CatalogBrowser00135')